## xBD Dataset Exploration & Preprocessing

Mirrors the structure of `qqb_preprocessing.ipynb`.

In this file: extract individual building crops from Mexico earthquake scenes,
map the 4-class labels to binary (no-damage = 0 / rest = 1),
and produce `xbd_test_buildings.csv` with one row per building.


In [1]:
import os
import json
import csv
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from PIL import Image
from shapely.wkt import loads as wkt_loads

***0. Root paths***

In [2]:
TIER3_ROOT = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_tier3\tier3"
TRAIN_ROOT = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_train_images_labels_targets\train"

SOURCES = {
    "tier3": (
        os.path.join(TIER3_ROOT, "images"),
        os.path.join(TIER3_ROOT, "labels"),
    ),
    "train": (
        os.path.join(TRAIN_ROOT, "images"),
        os.path.join(TRAIN_ROOT, "labels"),
    ),
}

BASE_OUTPUT = Path(r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\xbd_dataset")
CROPS_DIR   = BASE_OUTPUT / "xbd_building_crops"
OUTPUT_CSV  = BASE_OUTPUT / "xbd_test_buildings.csv"
CROPS_DIR.mkdir(parents=True, exist_ok=True)

for source, (img_dir, lbl_dir) in SOURCES.items():
    img_ok = os.path.isdir(img_dir)
    lbl_ok = os.path.isdir(lbl_dir)
    print(f"{source}  images: {'OK' if img_ok else 'NOT FOUND'}  "
          f"labels: {'OK' if lbl_ok else 'NOT FOUND'}")

tier3  images: OK  labels: OK
train  images: OK  labels: OK


***1. File counts per source***

In [3]:
for source, (img_dir, lbl_dir) in SOURCES.items():
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".png")]
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith(".json")]
    print(f"{source}:  {len(img_files)} images,  {len(lbl_files)} labels")
    print(f"  sample image : {img_files[0]}")
    print(f"  sample label : {lbl_files[0]}")

tier3:  12738 images,  12738 labels
  sample image : joplin-tornado_00000000_post_disaster.png
  sample label : joplin-tornado_00000000_post_disaster.json
train:  5598 images,  5598 labels
  sample image : guatemala-volcano_00000000_post_disaster.png
  sample label : guatemala-volcano_00000000_post_disaster.json


***2. Inspect one sample GeoJSON label***

In [4]:
def find_first_post_json(labels_dir):
    for fname in os.listdir(labels_dir):
        if "post_disaster" in fname and fname.endswith(".json"):
            return os.path.join(labels_dir, fname)
    return None

sample_json = find_first_post_json(SOURCES["tier3"][1])
with open(sample_json) as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print("Metadata:", data["metadata"])
features = data["features"]["xy"]
print(f"\nNumber of labelled buildings: {len(features)}")
print("First feature keys  :", list(features[0].keys()))
print("First feature props :", features[0].get("properties", {}))

Top-level keys: ['features', 'metadata']
Metadata: {'sensor': 'WORLDVIEW02', 'provider_asset_type': 'WORLDVIEW02', 'gsd': 2.35255861282349, 'capture_date': '2011-05-29T17:29:25.433Z', 'off_nadir_angle': 28.4302768707275, 'pan_resolution': 0.585882604122162, 'sun_azimuth': 143.603851318359, 'sun_elevation': 71.8531799316406, 'target_azimuth': 193.251083374023, 'disaster': 'joplin-tornado', 'disaster_type': 'wind', 'catalog_id': '103001000A285500', 'original_width': 1024, 'original_height': 1024, 'width': 1024, 'height': 1024, 'id': 'MjY0Mzk4Mg.vo5dFUhTfoVWnKNLRID8rEkx0A0', 'img_name': 'joplin-tornado_00000000_post_disaster.png'}

Number of labelled buildings: 185
First feature keys  : ['properties', 'wkt']
First feature props : {'feature_type': 'building', 'subtype': 'destroyed', 'uid': 'b6aa615e-56b2-458a-b404-d094b64dcad3'}


***3. Discover all disaster types***

In [5]:
def disaster_name(fname):
    parts = fname.split("_")
    return "_".join(parts[:-3])

disaster_counts = Counter()
for source, (img_dir, _) in SOURCES.items():
    for fname in os.listdir(img_dir):
        if fname.endswith(".png") and "post_disaster" in fname:
            disaster_counts[disaster_name(fname)] += 1

print(f"Total unique disaster types: {len(disaster_counts)}\n")
print(f"{'Disaster':<42} Post-disaster images")
print("-" * 62)
for name, count in sorted(disaster_counts.items()):
    print(f"{name:<42} {count}")

Total unique disaster types: 19

Disaster                                   Post-disaster images
--------------------------------------------------------------
guatemala-volcano                          18
hurricane-florence                         319
hurricane-harvey                           319
hurricane-matthew                          238
hurricane-michael                          343
joplin-tornado                             149
lower-puna-volcano                         291
mexico-earthquake                          121
midwest-flooding                           279
moore-tornado                              227
nepal-flooding                             619
palu-tsunami                               113
pinery-bushfire                            1845
portugal-wildfire                          1869
santa-rosa-wildfire                        226
socal-fire                                 823
sunda-tsunami                              148
tuscaloosa-tornado                      

***4. Label mapping and extraction parameters***

We extract individual building crops from Mexico earthquake scenes only.

Label mapping:
- `no-damage`     → 0 (intact)
- `minor-damage`  → 1 (damaged)
- `major-damage`  → 1 (damaged)
- `destroyed`     → 1 (damaged)
- `un-classified` → skipped

MIN_SIZE = 10px: buildings smaller than 10×10 pixels are skipped.
PADDING  = 10px: context added around each building bounding box.

In [6]:
# Only Mexico earthquake — same disaster type as QQB training data
TARGET_DISASTER = "mexico-earthquake"

LABEL_MAP = {
    "no-damage":     0,
    "minor-damage":  1,
    "major-damage":  1,
    "destroyed":     1,
    "un-classified": None,   # excluded
}

PADDING  = 10    # pixels of context around each building
MIN_SIZE = 10    # minimum crop dimension in pixels

print(f"Target disaster: {TARGET_DISASTER}")
print(f"Label mapping:   {LABEL_MAP}")
print(f"Padding:         {PADDING}px")
print(f"Min crop size:   {MIN_SIZE}px")

Target disaster: mexico-earthquake
Label mapping:   {'no-damage': 0, 'minor-damage': 1, 'major-damage': 1, 'destroyed': 1, 'un-classified': None}
Padding:         10px
Min crop size:   10px


***5. Extract individual building crops***

In [7]:
def get_pixel_bbox(xy_wkt):
    """Returns (xmin, ymin, xmax, ymax) from a WKT polygon in pixel coords."""
    polygon = wkt_loads(xy_wkt)
    return polygon.bounds


def crop_building(scene, xmin, ymin, xmax, ymax, padding, scene_w, scene_h):
    """Crops a building from the scene with padding, clamped to scene bounds."""
    x0 = max(0, int(xmin) - padding)
    y0 = max(0, int(ymin) - padding)
    x1 = min(scene_w, int(xmax) + padding)
    y1 = min(scene_h, int(ymax) + padding)
    return scene.crop((x0, y0, x1, y1))


rows        = []
n_intact    = 0
n_damaged   = 0
n_skipped   = 0
n_too_small = 0
scene_count = 0

for source, (img_dir, lbl_dir) in SOURCES.items():
    for fname in sorted(os.listdir(img_dir)):
        if "post_disaster" not in fname or not fname.endswith(".png"):
            continue
        if disaster_name(fname) != TARGET_DISASTER:
            continue

        scene_path = Path(img_dir) / fname
        json_path  = Path(lbl_dir) / fname.replace(".png", ".json")

        if not scene_path.exists() or not json_path.exists():
            print(f"  WARNING: missing files for {fname}")
            continue

        scene   = Image.open(scene_path).convert("RGB")
        scene_w, scene_h = scene.size

        with open(json_path) as f:
            data = json.load(f)

        meta      = data.get("metadata", {})
        buildings = data["features"]["xy"]

        for building in buildings:
            props   = building["properties"]
            subtype = props.get("subtype", "un-classified")
            uid     = props.get("uid", "unknown")
            label   = LABEL_MAP.get(subtype, None)

            if label is None:
                n_skipped += 1
                continue

            try:
                xmin, ymin, xmax, ymax = get_pixel_bbox(building["wkt"])
            except Exception:
                n_skipped += 1
                continue

            if (xmax - xmin) < MIN_SIZE or (ymax - ymin) < MIN_SIZE:
                n_too_small += 1
                continue

            crop = crop_building(scene, xmin, ymin, xmax, ymax,
                                  PADDING, scene_w, scene_h)

            crop_path = CROPS_DIR / f"{uid}.png"
            crop.save(crop_path)

            rows.append({
                "path":         str(crop_path),
                "label":        label,
                "disaster":     TARGET_DISASTER,
                "subtype":      subtype,
                "uid":          uid,
                "scene_path":   str(scene_path),
                "sun_azimuth":  meta.get("sun_azimuth",  150.0),
                "sun_elevation":meta.get("sun_elevation", 32.0),
            })

            if label == 0:
                n_intact  += 1
            else:
                n_damaged += 1

        scene_count += 1
        if scene_count % 20 == 0:
            print(f"  Processed {scene_count} scenes — "
                  f"crops: {n_intact + n_damaged} "
                  f"(intact: {n_intact}, damaged: {n_damaged})")

print(f"\nDone processing {scene_count} scenes.")

  Processed 20 scenes — crops: 5571 (intact: 5559, damaged: 12)
  Processed 40 scenes — crops: 11516 (intact: 11460, damaged: 56)
  Processed 60 scenes — crops: 16619 (intact: 16535, damaged: 84)
  Processed 80 scenes — crops: 20558 (intact: 20456, damaged: 102)
  Processed 100 scenes — crops: 26046 (intact: 25930, damaged: 116)
  Processed 120 scenes — crops: 30586 (intact: 30461, damaged: 125)

Done processing 121 scenes.


***6. Save CSV and print summary***

In [8]:
df_buildings = pd.DataFrame(rows)
df_buildings.to_csv(OUTPUT_CSV, index=False)

print("Extraction complete.")
print(f"  Intact buildings:    {n_intact:,}")
print(f"  Damaged buildings:   {n_damaged:,}")
print(f"  Total:               {n_intact + n_damaged:,}")
print(f"  Ratio:               {n_intact / max(n_damaged, 1):.1f}:1")
print(f"  Skipped (unclass):   {n_skipped:,}")
print(f"  Skipped (too small): {n_too_small:,}")
print(f"\nSubtype breakdown of damaged:")
print(df_buildings[df_buildings['label']==1]['subtype'].value_counts().to_string())
print(f"\nCrops saved → {CROPS_DIR}")
print(f"CSV saved   → {OUTPUT_CSV}")

Extraction complete.
  Intact buildings:    30,461
  Damaged buildings:   125
  Total:               30,586
  Ratio:               243.7:1
  Skipped (unclass):   75
  Skipped (too small): 1,610

Subtype breakdown of damaged:
subtype
minor-damage    105
major-damage     18
destroyed         2

Crops saved → C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\xbd_dataset\xbd_building_crops
CSV saved   → C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\xbd_dataset\xbd_test_buildings.csv
